# `apptbl-magistro-subsets`: AI & offshoring OLS treatment-effect error by country, arm, and attribute type

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ASSET_NAME = "apptbl-magistro-subsets"
ASSET_DIR = Path.cwd()
PAPER_DIR = ASSET_DIR.parent
if not (PAPER_DIR / "microdata_analysis.py").is_file():
    raise FileNotFoundError(f"Open this notebook from manuscript-training-assets/; {PAPER_DIR} has no microdata_analysis.py")
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

import microdata_analysis as ma

print(f"Outputs go to {ASSET_DIR / ASSET_NAME}.*")

Outputs go to /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/manuscript-training-assets/apptbl-magistro-subsets.*


## Run design

In [2]:
RUN_ROOTS = (PAPER_DIR / "runs-rorqual", PAPER_DIR / "runs-nibi")
MODELS = ("kev-0.8b", "kev-4b", "kev-9b")
BASE_SLUG = "row1_base_untrained"
TRAINED_SLUGS = ("row2b_wu_ce_oracle", "row3_berke_ce_only", "row5_full_berke_wu_ce_oracle")
VALIDATORS = ("kriner2026", "grummon2025")

ctx = ma.load_microdata_context(
    run_roots=RUN_ROOTS,
    models=MODELS,
    base_slug=BASE_SLUG,
    trained_slugs=TRAINED_SLUGS,
    validators=VALIDATORS,
)
display(ctx.status_grid)

,kev-0.8b,kev-4b,kev-9b
regime,,,
row1_base_untrained,step 0 [test_best ✓],step 0 [test_best ✓],step 0 [test_best ✓]
row2b_wu_ce_oracle,finished (@640/1395) [test_best ✓],finished (@416/1395) [test_best ✓],finished (@928/1395) [test_best ✓]
row3_berke_ce_only,finished (@1152/2730) [test_best ✓],finished (@544/2730) [test_best ✓],finished (@256/2730) [test_best ✓]
row5_full_berke_wu_ce_oracle,finished (@1600/4125) [test_best ✓],finished (@1472/4125) [test_best ✓],finished (@1600/4125) [test_best ✓]


## Labels and formatting (edit wording here)

In [3]:
MODEL_LABELS = {"kev-0.8b": "KEV-0.8B", "kev-4b": "KEV-4B", "kev-9b": "KEV-9B"}

# Row order follows this dict.
REGIME_LABELS = {
    BASE_SLUG: "Off-the-shelf",
    "row2b_wu_ce_oracle": "Electricity",
    "row3_berke_ce_only": "Drones",
    "row5_full_berke_wu_ce_oracle": "Drones + Electricity",
}

STUB_HEADERS = ("Model", "Trained on")

US_REGRESSIONS = ("magistro2025/us_ai", "magistro2025/us_offshoring")
CANADA_REGRESSIONS = ("magistro2025/ca_ai", "magistro2025/ca_offshoring")
ALL_REGRESSIONS = US_REGRESSIONS + CANADA_REGRESSIONS

# Column label -> (regressions to include, which terms: "all", "price", "non-price"). {n} becomes the number of slopes.
SUBSET_COLUMNS = {
    "All ({n})": (ALL_REGRESSIONS, "all"),
    "US ({n})": (US_REGRESSIONS, "all"),
    "Canada ({n})": (CANADA_REGRESSIONS, "all"),
    "US AI ({n})": (("magistro2025/us_ai",), "all"),
    "US Off. ({n})": (("magistro2025/us_offshoring",), "all"),
    "CA AI ({n})": (("magistro2025/ca_ai",), "all"),
    "CA Off. ({n})": (("magistro2025/ca_offshoring",), "all"),
    "Price ({n})": (ALL_REGRESSIONS, "price"),
    "Jobs ({n})": (ALL_REGRESSIONS, "non-price"),
}
PRICE_MARKER = "| Price:"

MSE_SCALE = 1000.0
MSE_DECIMALS = 2

## Build the table

In [4]:
def escape_markdown(text):
    """Escape $ and | so Pandoc reads them as text inside a pipe table."""
    escaped = text.replace("$", r"\$")
    escaped = escaped.replace("|", r"\|")
    return escaped


def pipe_table(headers, rows, column_rules):
    """Pandoc pipe table: header row, rule row (e.g. ':---' left, '---:' right), then one line per row."""
    if len(column_rules) != len(headers):
        raise ValueError(f"{len(column_rules)} column rules for {len(headers)} columns")
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("|" + "|".join(column_rules) + "|")
    for row in rows:
        if len(row) != len(headers):
            raise ValueError(f"Row has {len(row)} cells, expected {len(headers)}: {row}")
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines) + "\n"

In [5]:
def subset_mask(betas, regressions, term_group):
    in_regressions = betas["model"].isin(regressions)
    is_price = betas.index.str.contains(PRICE_MARKER, regex=False)
    if term_group == "all":
        return in_regressions
    if term_group == "price":
        return in_regressions & is_price
    if term_group == "non-price":
        return in_regressions & ~is_price
    raise ValueError(f"Unknown term group {term_group!r}")


reference_betas = ctx.magistro_betas_by_model[MODELS[0]][BASE_SLUG]
column_headers = []
for column_template, (regressions, term_group) in SUBSET_COLUMNS.items():
    mask = subset_mask(reference_betas, regressions, term_group)
    if not mask.any():
        raise ValueError(f"Empty subset for column {column_template!r}")
    column_headers.append(column_template.replace("{n}", str(int(mask.sum()))))

# Bold marks the lowest value per model and subset.
rows = []
for model_slug in MODELS:
    mse_by_regime = {}
    for regime_slug in REGIME_LABELS:
        betas = ctx.magistro_betas_by_model[model_slug][regime_slug]
        squared_errors = (betas["model_pop_beta"] - betas["human_pop_beta"]) ** 2
        subset_mses = []
        for regressions, term_group in SUBSET_COLUMNS.values():
            mask = subset_mask(betas, regressions, term_group)
            subset_mses.append(float(squared_errors.loc[mask].mean()))
        mse_by_regime[regime_slug] = subset_mses
    mse_df = pd.DataFrame(mse_by_regime, index=column_headers).T
    best_slugs = mse_df.idxmin()

    for regime_index, (regime_slug, regime_label) in enumerate(REGIME_LABELS.items()):
        model_cell = MODEL_LABELS[model_slug] if regime_index == 0 else ""
        row = [model_cell, regime_label]
        for column_header in column_headers:
            text = f"{mse_df.loc[regime_slug, column_header] * MSE_SCALE:.{MSE_DECIMALS}f}"
            if best_slugs[column_header] == regime_slug:
                text = f"**{text}**"
            row.append(text)
        rows.append(row)

headers = [*STUB_HEADERS, *column_headers]
column_rules = [":---"] * len(STUB_HEADERS) + ["---:"] * len(column_headers)
table_md = pipe_table(headers, rows, column_rules)
display(pd.DataFrame(rows, columns=headers))

,Model,Trained on,All (32),US (16),Canada (16),US AI (8),US Off. (8),CA AI (8),CA Off. (8),Price (8),Jobs (24)
0,KEV-0.8B,Off-the-shelf,33.92,41.48,26.35,50.62,32.34,24.20,28.51,77.60,19.36
1,,Electricity,29.60,35.59,23.62,42.21,28.96,20.16,27.08,71.71,15.57
2,,Drones,34.41,41.14,27.67,49.82,32.46,25.25,30.10,83.27,18.12
3,,Drones + Electricity,**26.61**,**32.17**,**21.06**,**38.38**,**25.96**,**17.60**,**24.51**,**67.37**,**13.03**
4,KEV-4B,Off-the-shelf,99.01,93.76,104.27,92.30,95.22,96.88,111.65,16.09,126.66
5,,Electricity,113.26,106.45,120.08,101.87,111.03,112.91,127.26,11.67,147.13
6,,Drones,74.69,67.29,82.09,60.96,73.62,74.73,89.45,6.21,97.52
7,,Drones + Electricity,**63.98**,**56.05**,**71.90**,**53.45**,**58.66**,**70.30**,**73.49**,**5.66**,**83.42**
8,KEV-9B,Off-the-shelf,82.89,75.34,90.43,67.71,82.98,80.55,100.31,**21.31**,103.41
9,,Electricity,95.68,83.68,107.69,71.35,96.01,98.52,116.86,37.05,115.23


## Save `apptbl-magistro-subsets.md` (included by the manuscript)

In [6]:
table_path = ASSET_DIR / f"{ASSET_NAME}.md"
table_path.write_text(table_md, encoding="utf-8")
print(table_md)
print(f"Saved {table_path}")

| Model | Trained on | All (32) | US (16) | Canada (16) | US AI (8) | US Off. (8) | CA AI (8) | CA Off. (8) | Price (8) | Jobs (24) |
|:---|:---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| KEV-0.8B | Off-the-shelf | 33.92 | 41.48 | 26.35 | 50.62 | 32.34 | 24.20 | 28.51 | 77.60 | 19.36 |
|  | Electricity | 29.60 | 35.59 | 23.62 | 42.21 | 28.96 | 20.16 | 27.08 | 71.71 | 15.57 |
|  | Drones | 34.41 | 41.14 | 27.67 | 49.82 | 32.46 | 25.25 | 30.10 | 83.27 | 18.12 |
|  | Drones + Electricity | **26.61** | **32.17** | **21.06** | **38.38** | **25.96** | **17.60** | **24.51** | **67.37** | **13.03** |
| KEV-4B | Off-the-shelf | 99.01 | 93.76 | 104.27 | 92.30 | 95.22 | 96.88 | 111.65 | 16.09 | 126.66 |
|  | Electricity | 113.26 | 106.45 | 120.08 | 101.87 | 111.03 | 112.91 | 127.26 | 11.67 | 147.13 |
|  | Drones | 74.69 | 67.29 | 82.09 | 60.96 | 73.62 | 74.73 | 89.45 | 6.21 | 97.52 |
|  | Drones + Electricity | **63.98** | **56.05** | **71.90** | **53.45** | **58.66** | **70.30** | **73.49**